# Evaluation & Metrics Engineer
### COVID-19 Radiography Classification — Model Performance Report

**Task 5**: Compute performance metrics (accuracy, F1, precision, recall, etc.), compare CNN vs FFNN models, and create result tables.

**Classes**: COVID | Lung_Opacity | Normal | Viral_Pneumonia

## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)
import warnings
warnings.filterwarnings('ignore')

# Class names (must match dataset class_to_idx order)
CLASS_NAMES = ['COVID', 'Lung_Opacity', 'Normal', 'Viral_Pneumonia']
NUM_CLASSES  = 4

print("All imports successful!")

## 2. Load Trained Models

> **Note:** Make sure `cnn_model.pth` and `ffnn_model.pth` are saved by the CNN & FFNN engineers.
> Run their training notebooks first, then come back here.

In [ ]:
# ─── FFNN Architecture (must match the one in FFNN notebook) ───────────────────
class FFNN_COVID(nn.Module):
    def __init__(self, num_classes=4):
        super(FFNN_COVID, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU(inplace=True)
        self.dropout2d = nn.Dropout2d(0.25)
        self.flatten   = nn.Flatten()
        self.fc1 = nn.Linear(8192, 4096)
        self.fc2 = nn.Linear(4096, 1024)
        self.fc3 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, num_classes)
        self.bn1 = nn.BatchNorm1d(4096)
        self.bn2 = nn.BatchNorm1d(1024)
        self.bn3 = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.relu(self.conv1(x)); x = self.pool(x); x = self.dropout2d(x)
        x = self.relu(self.conv2(x)); x = self.pool(x); x = self.dropout2d(x)
        x = self.relu(self.conv3(x))
        x = F.adaptive_avg_pool2d(x, (8, 8))
        x = self.flatten(x)
        x = self.dropout(self.relu(self.bn1(self.fc1(x))))
        x = self.dropout(self.relu(self.bn2(self.fc2(x))))
        x = self.dropout(self.relu(self.bn3(self.fc3(x))))
        return self.fc4(x)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# ─── Load saved weights ──────────────────────────────────────────────────────
# CNN (Keras / TF)
import tensorflow as tf
cnn_model = tf.keras.models.load_model('cnn_model.h5')
print("CNN model loaded!")

# FFNN (PyTorch)
ffnn_model = FFNN_COVID(num_classes=NUM_CLASSES).to(DEVICE)
ffnn_model.load_state_dict(torch.load('ffnn_model.pth', map_location=DEVICE))
ffnn_model.eval()
print("FFNN model loaded!")

## 3. Generate Predictions

We run both models on the **same test set** to ensure a fair comparison.
> `test_loader` comes from the shared data preprocessing notebook (run it first).

In [ ]:
# ─── Run data_preprocessing to get test_loader ──────────────────────────────
# %run data_preprocessing_01.ipynb   # Uncomment if running standalone

# ─── CNN predictions (TF/Keras) ─────────────────────────────────────────────
def predict_cnn(dataloader):
    """Generate CNN predictions via torch->tf generator."""
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in dataloader:
        imgs_np = images.permute(0, 2, 3, 1).numpy()   # (B,H,W,C)
        probs   = cnn_model.predict(imgs_np, verbose=0)
        preds   = np.argmax(probs, axis=1)
        all_probs.append(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.vstack(all_probs)

# ─── FFNN predictions (PyTorch) ─────────────────────────────────────────────
def predict_ffnn(dataloader):
    """Generate FFNN predictions."""
    all_preds, all_labels, all_probs = [], [], []
    ffnn_model.eval()
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(DEVICE)
            logits = ffnn_model(images)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            preds  = np.argmax(probs, axis=1)
            all_probs.append(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.vstack(all_probs)

print("Running CNN predictions...")
cnn_true, cnn_pred, cnn_probs = predict_cnn(test_loader)

print("Running FFNN predictions...")
ffnn_true, ffnn_pred, ffnn_probs = predict_ffnn(test_loader)

print(f"Test samples: {len(cnn_true)}")

## 4. Core Metrics Computation

In [ ]:
def compute_metrics(y_true, y_pred, y_probs, model_name):
    """Compute and print all evaluation metrics for a model."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1w  = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # AUC-ROC (one-vs-rest, macro)
    try:
        auc_score = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
    except Exception:
        auc_score = float('nan')

    print(f"\n{'='*55}")
    print(f"  {model_name} — Overall Metrics")
    print(f"{'='*55}")
    print(f"  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision (macro) : {prec:.4f}")
    print(f"  Recall    (macro) : {rec:.4f}")
    print(f"  F1-Score  (macro) : {f1:.4f}")
    print(f"  F1-Score (weighted): {f1w:.4f}")
    print(f"  AUC-ROC   (macro) : {auc_score:.4f}")
    print(f"{'='*55}")
    print(f"\n  Per-Class Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision (macro)': round(prec, 4),
        'Recall (macro)': round(rec, 4),
        'F1 (macro)': round(f1, 4),
        'F1 (weighted)': round(f1w, 4),
        'AUC-ROC': round(auc_score, 4)
    }

cnn_metrics  = compute_metrics(cnn_true,  cnn_pred,  cnn_probs,  'CNN  (Keras/TF)')
ffnn_metrics = compute_metrics(ffnn_true, ffnn_pred, ffnn_probs, 'FFNN (PyTorch)')

## 5. Comparison Table

In [ ]:
comparison_df = pd.DataFrame([cnn_metrics, ffnn_metrics]).set_index('Model')

# Highlight the best value per column
styled = comparison_df.style \
    .highlight_max(axis=0, color='#d4edda') \
    .format("{:.4f}") \
    .set_caption("CNN vs FFNN — Metrics Comparison (green = better)")

display(styled)
print("\n* Green cells indicate the model with the higher value for that metric.")

## 6. Confusion Matrices

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    annot = np.array(
        [[f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)" for j in range(cm.shape[1])]
          for i in range(cm.shape[0])]
    )

    sns.heatmap(
        cm_pct, annot=annot, fmt='', cmap=cmap,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5, linecolor='white',
        vmin=0, vmax=100, cbar_kws={'label': '% of true class'}
    )
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.tick_params(axis='x', rotation=25)
    ax.tick_params(axis='y', rotation=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrices — Test Set', fontsize=16, fontweight='bold', y=1.02)

plot_confusion_matrix(cnn_true,  cnn_pred,  'CNN (Keras/TF)',  axes[0], cmap='Blues')
plot_confusion_matrix(ffnn_true, ffnn_pred, 'FFNN (PyTorch)',  axes[1], cmap='Oranges')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices.png")

## 7. Per-Class Metrics Bar Chart

In [ ]:
def per_class_metrics(y_true, y_pred):
    p = precision_score(y_true, y_pred, average=None, zero_division=0)
    r = recall_score(y_true, y_pred, average=None, zero_division=0)
    f = f1_score(y_true, y_pred, average=None, zero_division=0)
    return p, r, f

cnn_p,  cnn_r,  cnn_f  = per_class_metrics(cnn_true,  cnn_pred)
ffnn_p, ffnn_r, ffnn_f = per_class_metrics(ffnn_true, ffnn_pred)

x = np.arange(NUM_CLASSES)
width = 0.18
metrics_labels = ['Precision', 'Recall', 'F1-Score']

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle('Per-Class Metrics: CNN vs FFNN', fontsize=15, fontweight='bold')

colors = {'CNN': '#4A90D9', 'FFNN': '#E87040'}

for ax, (cnn_vals, ffnn_vals, label) in zip(
    axes,
    [(cnn_p, ffnn_p, 'Precision'), (cnn_r, ffnn_r, 'Recall'), (cnn_f, ffnn_f, 'F1-Score')]
):
    bars1 = ax.bar(x - width/2, cnn_vals,  width, label='CNN',  color=colors['CNN'],  alpha=0.85)
    bars2 = ax.bar(x + width/2, ffnn_vals, width, label='FFNN', color=colors['FFNN'], alpha=0.85)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    # Value labels on bars
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: per_class_metrics.png")

## 8. ROC Curves (One-vs-Rest)

In [ ]:
from sklearn.preprocessing import label_binarize

def plot_roc_curves(y_true, y_probs, title, ax, colors_list):
    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors_list)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, color=color, label=f'{cls_name} (AUC = {roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
    ax.set_xlim([-0.01, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)

roc_colors = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ROC Curves (One-vs-Rest) — Test Set', fontsize=15, fontweight='bold')

plot_roc_curves(cnn_true,  cnn_probs,  'CNN (Keras/TF)',  axes[0], roc_colors)
plot_roc_curves(ffnn_true, ffnn_probs, 'FFNN (PyTorch)',  axes[1], roc_colors)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roc_curves.png")

## 9. Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.4)

# ── Radar / Summary bar ──────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0])

metric_keys = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)', 'AUC-ROC']
cnn_vals_bar  = [cnn_metrics[k]  for k in metric_keys]
ffnn_vals_bar = [ffnn_metrics[k] for k in metric_keys]

x_bar = np.arange(len(metric_keys))
bw = 0.35
b1 = ax.bar(x_bar - bw/2, cnn_vals_bar,  bw, label='CNN',  color='#4A90D9', alpha=0.85)
b2 = ax.bar(x_bar + bw/2, ffnn_vals_bar, bw, label='FFNN', color='#E87040', alpha=0.85)

ax.set_ylim(0, 1.18)
ax.set_xticks(x_bar)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC'], rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_title('Overall Metric Comparison', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8.5)

# ── Winner summary table ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.axis('off')

table_data  = []
col_labels  = ['Metric', 'CNN', 'FFNN', 'Winner']
for k in metric_keys:
    cv, fv = cnn_metrics[k], ffnn_metrics[k]
    winner = 'CNN ✓' if cv > fv else ('FFNN ✓' if fv > cv else 'Tie')
    table_data.append([k, f'{cv:.4f}', f'{fv:.4f}', winner])

tbl = ax2.table(
    cellText=table_data, colLabels=col_labels,
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.6)
# Color header
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor('#2C3E50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
# Color winner cells
for i, row in enumerate(table_data, start=1):
    if 'CNN' in row[3]:
        tbl[i, 3].set_facecolor('#C8E6C9')
    elif 'FFNN' in row[3]:
        tbl[i, 3].set_facecolor('#FFE0B2')

ax2.set_title('Head-to-Head Summary', fontsize=13, fontweight='bold', pad=12)

fig.suptitle('Model Evaluation Dashboard — CNN vs FFNN', fontsize=15, fontweight='bold', y=1.03)
plt.savefig('evaluation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: evaluation_dashboard.png")

## 10. Save Results to CSV

In [ ]:
comparison_df.to_csv('metrics_comparison.csv')
print("Saved: metrics_comparison.csv")
display(comparison_df)

---
## Notes for Report

| Metric | What it tells us |
|--------|------------------|
| **Accuracy** | Overall correct predictions / total samples |
| **Precision (macro)** | Of all predicted positives, how many were actually positive — averaged equally across classes |
| **Recall (macro)** | Of all actual positives, how many did we catch — averaged equally across classes |
| **F1 (macro)** | Harmonic mean of precision & recall — good when class sizes are unequal |
| **F1 (weighted)** | Same but weighted by class support — more representative for imbalanced datasets |
| **AUC-ROC** | Area under the ROC curve — 1.0 = perfect, 0.5 = random classifier |

> **Why macro F1?** The COVID-19 dataset has class imbalance (COVID < Lung_Opacity < Normal). Macro F1 treats each class equally, so it penalizes a model that ignores the minority class.